# Step 1 — Collect the Data
## Etapa 2.1 — Coleta e Documentação do Dataset Tabular

**Projeto:** Tech Challenge Fase 1 — Desafio B — Detecção de Pneumonia  
**Metodologia:** ML Life Cycle — Step 1 (Collect the Data)  

---

### Contexto do Problema

A pneumonia é uma infecção pulmonar grave que pode ser fatal, especialmente em populações vulneráveis (idosos, crianças, imunossuprimidos). O diagnóstico precoce é crítico: atrasos no tratamento aumentam significativamente a mortalidade.

O objetivo deste projeto é construir um modelo de Machine Learning capaz de **classificar o diagnóstico clínico de um paciente** com base em sinais vitais, sintomas e resultado de exames, distinguindo entre:
- **Pneumonia**
- **Edema Pulmonar** (pulmonary edema)
- **Atelectasia** (atelectasis)

Trata-se de um problema de **classificação multiclasse** com alta relevância clínica, onde erros de classificação têm impacto direto no tratamento do paciente.

> **Por que ML pode ajudar?**  
> Em ambientes de alta demanda (pronto-socorros, UTIs), modelos de triagem automatizada podem auxiliar o médico a priorizar casos graves, reduzindo o tempo até o diagnóstico definitivo.

---

## 1. Importações

In [2]:
import pandas as pd
import numpy as np

print('Bibliotecas carregadas com sucesso.')
print(f'pandas : {pd.__version__}')
print(f'numpy  : {np.__version__}')

Bibliotecas carregadas com sucesso.
pandas : 3.0.3
numpy  : 2.4.4


---

## 2. Proveniência e Documentação do Dataset

In [9]:
RANDOM_STATE = 42
DATASET_PATH = '../data/tabular/clinical_pneumonia_dataset.csv'

fonte = {
    'nome'           : 'Pneumonia Prediction Dataset',
    'url'            : 'https://www.kaggle.com/datasets/ajithdari/pneumonia-prediction-dataset',
    'autor'          : 'Ajith Dari (Kaggle)',
    'licenca'        : 'CC0: Public Domain',
    'data_download'  : '2026-05-12',
    'formato'        : 'CSV',
    'arquivo_local'  : 'data/tabular/clinical_pneumonia_dataset.csv',
    'descricao'      : (
        'Dataset clinico sintetico com notas de evolucao de pacientes internados. '
        'Cada registro representa uma nota clinica diaria de um paciente, contendo '
        'sinais vitais (saturacao de oxigenio), contagem de leucocitos, resultado '
        'de raio-X e sintomas binarios (febre, taquicardia, estertores). '
        'O label indica o diagnostico final: pneumonia, edema pulmonar ou atelectasia.'
    ),
    'coluna_target'  : 'true_label',
    'tipo_problema'  : 'Classificacao multiclasse (3 classes)',
    'total_registros': 1500,
    'total_features' : None,
    'total_pacientes': 297,
}

print('=== METADADOS DO DATASET ===')
print()
for chave, valor in fonte.items():
    print(f'  {chave:<22}: {valor}')

=== METADADOS DO DATASET ===

  nome                  : Pneumonia Prediction Dataset
  url                   : https://www.kaggle.com/datasets/ajithdari/pneumonia-prediction-dataset
  autor                 : Ajith Dari (Kaggle)
  licenca               : CC0: Public Domain
  data_download         : 2026-05-12
  formato               : CSV
  arquivo_local         : data/tabular/clinical_pneumonia_dataset.csv
  descricao             : Dataset clinico sintetico com notas de evolucao de pacientes internados. Cada registro representa uma nota clinica diaria de um paciente, contendo sinais vitais (saturacao de oxigenio), contagem de leucocitos, resultado de raio-X e sintomas binarios (febre, taquicardia, estertores). O label indica o diagnostico final: pneumonia, edema pulmonar ou atelectasia.
  coluna_target         : true_label
  tipo_problema         : Classificacao multiclasse (3 classes)
  total_registros       : 1500
  total_features        : None
  total_pacientes       : 297


---

## 3. Carregamento dos Dados

In [10]:
df = pd.read_csv(DATASET_PATH)

fonte['total_registros'] = df.shape[0]
fonte['total_features']  = df.shape[1] - 1
fonte['total_pacientes'] = df['patient_id'].nunique()

print(f'Dataset carregado com sucesso!')
print(f'  Linhas    : {df.shape[0]:,}')
print(f'  Colunas   : {df.shape[1]}')
print(f'  Pacientes : {df["patient_id"].nunique():,} unicos')

Dataset carregado com sucesso!
  Linhas    : 1,500
  Colunas   : 12
  Pacientes : 297 unicos


---

## 4. Visao Inicial dos Dados

In [11]:
df.head(3)

,patient_id,timestamp,note_sequence,clinical_note,fever,tachycardia,crackles,oxygen_saturation,wbc_count,chest_xray_result,true_label,uncertainty_score
0,PNE-6733,2022-06-11,0,Pyrexia. rapid heart rate. rales. hypoxemia. p...,1,1,1,93.5,7.5,consolidation,pneumonia,0.77
1,PNE-6733,2022-06-12,1,Fever. tachycardia. rales. spo2 < 94%. product...,1,1,1,93.5,7.5,consolidation,pneumonia,0.85
2,PNE-6733,2022-06-13,2,Raised body temp. tachycardia. crackles. spo2 ...,1,1,1,93.5,7.5,consolidation,pneumonia,0.96


**Observacao:** O dataset contem notas clinicas sequenciais por paciente (`note_sequence` 0 a 4). Cada linha representa uma observacao diaria do mesmo paciente. O split treino/teste devera ser feito por `patient_id` para evitar data leakage entre notas do mesmo paciente.

---

## 5. Tipos de Dados e Valores Nulos

In [12]:
print('=== INFORMACOES DAS COLUNAS ===')
print()
df.info()

=== INFORMACOES DAS COLUNAS ===

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   patient_id         1500 non-null   str    
 1   timestamp          1500 non-null   str    
 2   note_sequence      1500 non-null   int64  
 3   clinical_note      1500 non-null   str    
 4   fever              1500 non-null   int64  
 5   tachycardia        1500 non-null   int64  
 6   crackles           1500 non-null   int64  
 7   oxygen_saturation  1500 non-null   float64
 8   wbc_count          1500 non-null   float64
 9   chest_xray_result  1500 non-null   str    
 10  true_label         1500 non-null   str    
 11  uncertainty_score  1500 non-null   float64
dtypes: float64(3), int64(4), str(5)
memory usage: 140.8 KB


In [13]:
resumo_colunas = pd.DataFrame({
    'Tipo'  : df.dtypes,
    'Nulos' : df.isnull().sum(),
    'Unicos': df.nunique(),
    'Exemplo': df.iloc[0],
}).reset_index(names='Coluna')

display(resumo_colunas)

,Coluna,Tipo,Nulos,Unicos,Exemplo
0,patient_id,str,0,297,PNE-6733
1,timestamp,str,0,833,2022-06-11
2,note_sequence,int64,0,5,0
3,clinical_note,str,0,629,Pyrexia. rapid heart rate. rales. hypoxemia. p...
4,fever,int64,0,2,1
5,tachycardia,int64,0,2,1
6,crackles,int64,0,2,1
7,oxygen_saturation,float64,0,661,93.5
8,wbc_count,float64,0,646,7.5
9,chest_xray_result,str,0,5,consolidation


**Resultado:** Nenhum valor nulo em nenhuma coluna. O dataset esta completamente preenchido — sem necessidade de imputacao.

---

## 6. Distribuicao da Variavel Target

In [14]:
print('=== DISTRIBUICAO DO TARGET (true_label) ===')
print()
contagem_target = df['true_label'].value_counts()
display(contagem_target.to_frame(name='Contagem').assign(
    Percentual=lambda x: (x['Contagem'] / len(df) * 100).round(2).astype(str) + '%'
))

=== DISTRIBUICAO DO TARGET (true_label) ===



,Contagem,Percentual
true_label,,
pneumonia,500,33.33%
pulmonary edema,500,33.33%
atelectasis,500,33.33%


**Dataset perfeitamente balanceado:** cada uma das 3 classes possui exatamente 500 registros (33,33%). Isso e uma caracteristica do dataset sintetico. Na pratica clinica real, pneumonia seria a classe mais frequente. Para o projeto, o balanceamento elimina a necessidade de oversampling (SMOTE) ou `class_weight='balanced'` neste momento.

---

## 7. Analise das Features

In [15]:
print('=== FEATURES NUMERICAS (estatisticas descritivas) ===')
display(df[['oxygen_saturation', 'wbc_count', 'uncertainty_score']].describe().round(3))

=== FEATURES NUMERICAS (estatisticas descritivas) ===


,oxygen_saturation,wbc_count,uncertainty_score
count,1500.000,1500.000,1500.000
mean,94.592,8.460,0.661
std,1.860,1.271,0.192
min,90.520,6.000,0.300
25%,93.094,7.400,0.490
50%,94.644,8.500,0.700
75%,96.200,9.500,0.810
max,98.000,10.861,0.990


In [16]:
print('=== FEATURES BINARIAS (distribuicao) ===')
for col in ['fever', 'tachycardia', 'crackles']:
    contagem = df[col].value_counts()
    print(f'\n{col}:')
    for val, cnt in contagem.items():
        print(f'  {val} -> {cnt:>4} registros ({cnt/len(df)*100:.1f}%)')

=== FEATURES BINARIAS (distribuicao) ===

fever:
  1 -> 1136 registros (75.7%)
  0 ->  364 registros (24.3%)

tachycardia:
  1 ->  915 registros (61.0%)
  0 ->  585 registros (39.0%)

crackles:
  1 ->  811 registros (54.1%)
  0 ->  689 registros (45.9%)


In [9]:
print('=== chest_xray_result (resultado do raio-X) ===')
display(df['chest_xray_result'].value_counts().to_frame(name='Contagem'))

=== chest_xray_result (resultado do raio-X) ===


,Contagem
chest_xray_result,
infiltrate,321
normal,313
consolidation,312
opacity,280
effusion,274


In [10]:
print('=== ESTRUTURA TEMPORAL (notas por paciente) ===')
notas_por_paciente = df.groupby('patient_id')['note_sequence'].count()
display(notas_por_paciente.value_counts().sort_index().to_frame(name='Pacientes'))
print(f'\nnote_sequence valores unicos: {sorted(df["note_sequence"].unique())}')

=== ESTRUTURA TEMPORAL (notas por paciente) ===


,Pacientes
note_sequence,
5,294
10,3



note_sequence valores unicos: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


---

## 8. Mapeamento das Features para o Modelo

In [11]:
mapeamento_features = pd.DataFrame([
    {'Coluna': 'patient_id',        'Tipo': 'Identificador',     'Uso': 'DESCARTAR', 'Justificativa': 'Identificador unico — sem valor preditivo'},
    {'Coluna': 'timestamp',         'Tipo': 'Data/hora',         'Uso': 'DESCARTAR', 'Justificativa': 'Data da nota — sem valor preditivo clinico direto'},
    {'Coluna': 'note_sequence',     'Tipo': 'Sequencia',         'Uso': 'DESCARTAR', 'Justificativa': 'Numero da nota (0-4) — metadado de sequencia'},
    {'Coluna': 'clinical_note',     'Tipo': 'Texto livre',       'Uso': 'DESCARTAR', 'Justificativa': 'Texto nao estruturado — exige NLP, fora do escopo de ML classico'},
    {'Coluna': 'fever',             'Tipo': 'Binaria (0/1)',     'Uso': 'FEATURE',   'Justificativa': 'Sinal clinico: presenca ou ausencia de febre'},
    {'Coluna': 'tachycardia',       'Tipo': 'Binaria (0/1)',     'Uso': 'FEATURE',   'Justificativa': 'Sinal clinico: taquicardia e comum em infeccoes respiratorias'},
    {'Coluna': 'crackles',          'Tipo': 'Binaria (0/1)',     'Uso': 'FEATURE',   'Justificativa': 'Sinal de ausculta: estertores fortemente associados a pneumonia'},
    {'Coluna': 'oxygen_saturation', 'Tipo': 'Numerica continua', 'Uso': 'FEATURE',   'Justificativa': 'Saturacao de O2 — indicador do comprometimento respiratorio'},
    {'Coluna': 'wbc_count',         'Tipo': 'Numerica continua', 'Uso': 'FEATURE',   'Justificativa': 'Leucocitos — marcador laboratorial de infeccao/inflamacao'},
    {'Coluna': 'chest_xray_result', 'Tipo': 'Categorica',        'Uso': 'FEATURE',   'Justificativa': 'Resultado do raio-X — feature diagnostica principal (5 categorias)'},
    {'Coluna': 'true_label',        'Tipo': 'Categorica',        'Uso': 'TARGET',    'Justificativa': 'Diagnostico final: pneumonia / pulmonary edema / atelectasis'},
    {'Coluna': 'uncertainty_score', 'Tipo': 'Numerica continua', 'Uso': 'DESCARTAR', 'Justificativa': 'ALERTA: score do anotador — usar causaria DATA LEAKAGE'},
])

display(mapeamento_features)

,Coluna,Tipo,Uso,Justificativa
0,patient_id,Identificador,DESCARTAR,Identificador unico — sem valor preditivo
1,timestamp,Data/hora,DESCARTAR,Data da nota — sem valor preditivo clinico direto
2,note_sequence,Sequencia,DESCARTAR,Numero da nota (0-4) — metadado de sequencia
3,clinical_note,Texto livre,DESCARTAR,"Texto nao estruturado — exige NLP, fora do esc..."
4,fever,Binaria (0/1),FEATURE,Sinal clinico: presenca ou ausencia de febre
5,tachycardia,Binaria (0/1),FEATURE,Sinal clinico: taquicardia e comum em infeccoe...
6,crackles,Binaria (0/1),FEATURE,Sinal de ausculta: estertores fortemente assoc...
7,oxygen_saturation,Numerica continua,FEATURE,Saturacao de O2 — indicador do comprometimento...
8,wbc_count,Numerica continua,FEATURE,Leucocitos — marcador laboratorial de infeccao...
9,chest_xray_result,Categorica,FEATURE,Resultado do raio-X — feature diagnostica prin...


**Alerta sobre `uncertainty_score`:** Esta coluna representa a confianca do anotador no label atribuido. Inclui-la seria data leakage: o modelo aprenderia a usar a incerteza do diagnostico para prever o diagnostico — informacao inexistente em producao real. **Deve ser descartada.**

**Sobre a estrutura temporal:** 297 pacientes geraram 1.500 registros (~5 notas/paciente). O split deve ser feito por `patient_id` para garantir que todas as notas de um paciente estejam no mesmo conjunto.

---

## 9. Configuracao Definida para as Proximas Etapas

In [17]:
FEATURES_USAR     = ['fever', 'tachycardia', 'crackles', 'oxygen_saturation', 'wbc_count', 'chest_xray_result']
TARGET            = 'true_label'
COLUNAS_DESCARTAR = ['patient_id', 'timestamp', 'note_sequence', 'clinical_note', 'uncertainty_score']

print('=== CONFIGURACAO PARA O PRE-PROCESSAMENTO ===')
print()
print(f'  Features a usar   : {FEATURES_USAR}')
print(f'  Target            : {TARGET}')
print(f'  Colunas descartar : {COLUNAS_DESCARTAR}')
print(f'  Tipo do problema  : Classificacao multiclasse (3 classes)')
print(f'  Balanceamento     : Perfeitamente balanceado (33.3% por classe)')
print(f'  Estrategia split  : por patient_id (evitar leakage temporal)')

print(f'\n=== RESUMO FINAL ===')
print()
print(f'  Registros         : {len(df):,}')
print(f'  Pacientes unicos  : {df["patient_id"].nunique():,}')
print(f'  Features clinicas : {len(FEATURES_USAR)}')
print(f'  Classes no target : {df[TARGET].nunique()} {list(df[TARGET].unique())}')

=== CONFIGURACAO PARA O PRE-PROCESSAMENTO ===

  Features a usar   : ['fever', 'tachycardia', 'crackles', 'oxygen_saturation', 'wbc_count', 'chest_xray_result']
  Target            : true_label
  Colunas descartar : ['patient_id', 'timestamp', 'note_sequence', 'clinical_note', 'uncertainty_score']
  Tipo do problema  : Classificacao multiclasse (3 classes)
  Balanceamento     : Perfeitamente balanceado (33.3% por classe)
  Estrategia split  : por patient_id (evitar leakage temporal)

=== RESUMO FINAL ===

  Registros         : 1,500
  Pacientes unicos  : 297
  Features clinicas : 6
  Classes no target : 3 ['pneumonia', 'pulmonary edema', 'atelectasis']


---

## 10. Conclusoes da Etapa 2.1

### Dataset escolhido e justificativa

O **Pneumonia Prediction Dataset** (Kaggle — CC0) foi selecionado pelos seguintes criterios:
- Formato estruturado (CSV), adequado para ML classico
- Features clinicas reais e interpretaveis por profissionais de saude
- 1.500 registros de 297 pacientes — volume adequado para treino e avaliacao
- Dataset balanceado — 500 registros por classe
- Licenca CC0 (dominio publico) — sem restricoes de uso

### Coluna target

`true_label` — 3 valores: `pneumonia`, `pulmonary edema`, `atelectasis`.

### Decisoes tomadas nesta etapa

1. **Problema multiclasse** (nao binario): 3 condicoes clinicas distintas
2. **`uncertainty_score` descartado**: data leakage
3. **`clinical_note` descartada**: texto livre fora do escopo de ML classico
4. **Metadados descartados**: `patient_id`, `timestamp`, `note_sequence`
5. **6 features clinicas** selecionadas: fever, tachycardia, crackles, oxygen_saturation, wbc_count, chest_xray_result
6. **Split por `patient_id`**: para evitar leakage temporal entre notas do mesmo paciente
7. **Sem imputacao necessaria**: nenhum valor nulo no dataset

### Proximo passo

`02_eda.ipynb` — Analise Exploratorio: distribuicoes, correlacoes, boxplots por classe e heatmap.